# Number verification — Atmospheric Remote Sensing (USTH)

This notebook recomputes **independently** every number that appears in the lessons, slides, quizzes and calculators of the site and compares it with `../data/verified.json` (written by `build/verify_numbers.py`). The formulas come from the study guide *Satellite Remote Sensing of Atmospheric Composition*. The simple models (TIR, O₂ A-band, averaging-kernel simulation) are home-made illustrations, not measured data.

In [1]:
import json, math, numpy as np
from scipy.optimize import brentq
V = json.load(open('../data/verified.json'))
H,C,KB = 6.62607015e-34, 2.99792458e8, 1.380649e-23
def chk(name, a, b, rel=1e-6):
    ok = abs(a-b) <= rel*max(1,abs(b)); print(('OK  ' if ok else 'FAIL'), name, a, b); assert ok

## Ch.1 — Lifetime
τ = 1/(k[OH]); parallel sinks 1/τ = Σ 1/τᵢ; plume τ = L/U; decay e^(−t/τ).

In [2]:
k, oh = 2e-13, 1e6
tau = 1/(k*oh)
chk('CO lifetime days', tau/86400, V['co_oh']['tau_days'])
chk('parallel 2,3,6 d', 1/(1/2+1/3+1/6), V['par_example']['total'])
chk('plume tau h (L=50 km, U=5 m/s)', 50e3/5/3600, V['plume']['tau_h'])
# exponential fit on synthetic data
x = np.linspace(0,150e3,31); y = np.exp(-x/50e3); chk('fit e-fold', -1/np.polyfit(x,np.log(y),1)[0], 50e3, 1e-9)
chk('e^-1', math.exp(-1), V['decay']['e1'])
print('half-life/tau =', math.log(2))

OK   CO lifetime days 57.870370370370374 57.870370370370374
OK   parallel 2,3,6 d 1.0000000000000002 1.0
OK   plume tau h (L=50 km, U=5 m/s) 2.7777777777777777 2.7777777777777777
OK   fit e-fold 49999.999999999985 50000.0
OK   e^-1 0.36787944117144233 0.36787944117144233
half-life/tau = 0.6931471805599453


## Ch.2 — Beer–Lambert, Planck, Wien, Stefan–Boltzmann, Rayleigh

In [3]:
tv, sza = 0.5, 60
ts = tv/math.cos(math.radians(sza)); chk('slant tau', ts, V['beer']['tau_slant']); chk('T', math.exp(-ts), V['beer']['T'])
B = lambda lam,T: 2*H*C**2/lam**5/(math.exp(H*C/(lam*KB*T))-1)
for name,T in [('sun',5778),('earth',288)]:
    from scipy.optimize import minimize_scalar
    lmax = minimize_scalar(lambda l: -B(l,T), bounds=(0.1e-6,60e-6), method='bounded', options=dict(xatol=1e-12)).x*1e6
    print(name,'numeric lambda_max (um)', round(lmax,4),' Wien', round(2898/T,4)); assert abs(lmax-2898/T)/(2898/T) < 0.01
chk('Stefan 288 K', 5.670374419e-8*288**4, V['stefan']['M_earth'], 1e-9)
chk('Rayleigh 450 vs 650', (650/450)**4, V['rayleigh_450_650'])

OK   slant tau 0.9999999999999998 0.9999999999999998
OK   T 0.3678794411714424 0.3678794411714424
sun numeric lambda_max (um) 0.5015  Wien 0.5016
earth numeric lambda_max (um) 10.0617  Wien 10.0625
OK   Stefan 288 K 390.1051535418615 390.1051535418615
OK   Rayleigh 450 vs 650 4.353147386069196 4.353147386069196


## Ch.3 — AMF, VCD, Dobson Unit

In [4]:
amf = 1/math.cos(math.radians(60)) + 1/math.cos(math.radians(30))
chk('AMF', amf, V['amf']['amf']); chk('VCD', (8e15-3e15)/amf, V['amf']['vcd'], 1e-9)
print('VCD in DU if O3:', (8e15-3e15)/amf/2.687e16)
chk('300 DU', 300*2.687e16, V['DU']['example_300DU'], 1e-9)

OK   AMF 3.154700538379251 3.154700538379251
OK   VCD 1584936490538903.8 1584936490538903.8
VCD in DU if O3: 0.05898535506285463
OK   300 DU 8.061e+18 8.061e+18


## Ch.5 — Optimal estimation: averaging kernel and DOFS
12-level model: Gaussian K of width w, S_ε = σ_ε² I, S_a correlated as exp(−|i−j|/L). A = (KᵀS_ε⁻¹K + S_a⁻¹)⁻¹KᵀS_ε⁻¹K; DOFS = tr(A) = sum of eigenvalues.

In [5]:
def oe(N=12,w=3.0,se=0.1,sa=1.0,Lc=3.0):
    z=np.arange(N); K=np.exp(-0.5*((z[None,:]-z[:,None])/w)**2); Sa=sa**2*np.exp(-np.abs(z[:,None]-z[None,:])/Lc)
    Sei=np.eye(N)/se**2; M=np.einsum('ij,jk->ik',np.einsum('ij,jk->ik',K.T,Sei),K)+np.linalg.inv(Sa)
    G=np.einsum('ij,jk->ik',np.linalg.inv(M),np.einsum('ij,jk->ik',K.T,Sei)); A=np.einsum('ij,jk->ik',G,K); return A
for name,p in dict(ideal=(0.3,0.01),realistic=(3,0.1),noisy=(3,1.0)).items():
    A=oe(12,*p); chk('DOFS '+name, np.trace(A), V['oe'][name]['dofs'], 1e-9); chk('sum eig '+name, np.linalg.eigvals(A).real.sum(), np.trace(A), 1e-9)
    assert np.allclose(A, np.array(V['oe'][name]['A']), atol=1e-9)
print('A recomputed: all 12x12 matrices of the 3 cases match')

OK   DOFS ideal 11.99645488442944 11.99645488442944
OK   sum eig ideal 11.99645488442944 11.99645488442944
OK   DOFS realistic 4.341169832865099 4.341169832865099
OK   sum eig realistic 4.3411698328650985 4.341169832865099
OK   DOFS noisy 2.89276948595352 2.89276948595352
OK   sum eig noisy 2.892769485953522 2.89276948595352
A recomputed: all 12x12 matrices of the 3 cases match


## Ch.6 — Ångström

In [6]:
a=math.log(0.60/0.15)/math.log(870/440); chk('alpha', a, V['angstrom']['alpha']); chk('AOD550', 0.60*(550/440)**(-a), V['angstrom']['tau550'])
ls=np.array([440,500,675,870,1020.]); t=0.6*(ls/440)**(-a); chk('regression alpha', -np.polyfit(np.log(ls),np.log(t),1)[0], a, 1e-9)

OK   alpha 2.033529077117641 2.033529077117641
OK   AOD550 0.3811377107074469 0.3811377107074469
OK   regression alpha 2.0335290771176373 2.033529077117641


## Ch.7 — Size parameter and extinction efficiency Q_ext (own Lorenz–Mie code, Bohren–Huffman)

In [7]:
xp=lambda r,l:2*math.pi*r/l
chk('x N2',xp(2e-4,.55),V['size']['N2']); chk('x sulfate',xp(.2,.55),V['size']['sulfate']); chk('x dust',xp(5,.55),V['size']['dust'])
def qext(m,x):
    nmax=int(x+4*x**(1/3)+2); mx=m*x; nmx=int(max(nmax,abs(mx))+16); D=np.zeros(nmx+1,dtype=complex)
    for n in range(nmx,0,-1): D[n-1]=n/mx-1/(D[n]+n/mx)
    psi0,psi1=math.cos(x),math.sin(x); chi0,chi1=-math.sin(x),math.cos(x); xi1=complex(psi1,-chi1); q=0
    for n in range(1,nmax+1):
        psi=(2*n-1)/x*psi1-psi0; chi=(2*n-1)/x*chi1-chi0; xi=complex(psi,-chi)
        a_=((D[n]/m+n/x)*psi-psi1)/((D[n]/m+n/x)*xi-xi1); b_=((D[n]*m+n/x)*psi-psi1)/((D[n]*m+n/x)*xi-xi1)
        q+=(2*n+1)*(a_.real+b_.real); psi0,psi1,chi0,chi1,xi1=psi1,psi,chi1,chi,xi
    return 2/x**2*q
for x in [0.1,0.5,1,2,5,10,20,50,100,200]: chk('Qext x=%g'%x, qext(1.33,x), V['mie_qext_m133'][str(x)], 1e-9)
m=1.33;xr=0.05;qr=8/3*xr**4*abs((m**2-1)/(m**2+2))**2; print('Rayleigh check x=0.05:', qext(m,xr), qr); assert abs(qext(m,xr)-qr)/qr<0.02

OK   x N2 0.002284794657156213 0.002284794657156213
OK   x sulfate 2.2847946571562128 2.2847946571562128
OK   x dust 57.119866428905326 57.119866428905326
OK   Qext x=0.1 1.109062536221262e-05 1.109062536221262e-05
OK   Qext x=0.5 0.006773139883838112 0.006773139883838112
OK   Qext x=1 0.09392400121407181 0.09392400121407181
OK   Qext x=2 0.7129483218556671 0.7129483218556671
OK   Qext x=5 3.5910329236295633 3.5910329236295633
OK   Qext x=10 2.2065487101846193 2.2065487101846193
OK   Qext x=20 2.1401071524274493 2.1401071524274493
OK   Qext x=50 1.9798862486569728 1.9798862486569728
OK   Qext x=100 2.1010699120374077 2.1010699120374077
OK   Qext x=200 2.0556193568075147 2.0556193568075147
Rayleigh check x=0.05: 6.935521560573177e-07 6.936801105674503e-07


## Ch.8 — Thermal infrared and O₂ A-band (simple models)

In [8]:
pl_inv=lambda lam,Bv: H*C/(lam*KB)/math.log(1+2*H*C**2/(lam**5*Bv))
def bt(Ts,z,tau,lam=10e-6,lap=6.5):
    Tl=Ts-lap*z; I=B(lam,Ts)*math.exp(-tau)+B(lam,Tl)*(1-math.exp(-tau)); return pl_inv(lam,I),Tl
b1,_=bt(300,1,.5); b5,_=bt(300,5,.5); chk('dBT 1 km',300-b1,V['tir']['z1']['dBT'],1e-9); chk('dBT 5 km',300-b5,V['tir']['z5']['dBT'],1e-9)
t_low=brentq(lambda t:(300-bt(300,1,t)[0])-(300-bt(300,5,.1)[0]),0.001,20); chk('degenerate tau (1 km)',t_low,V['tir_degenerate']['low_thick']['tau'],1e-8)
for z in (1,3,5,8): chk('O2 above %d km'%z, math.exp(-z/7), V['oxygen_above'][str(z)])
print('The 5 km layer gives a larger dBT by a factor of', round((300-b5)/(300-b1),2))

OK   dBT 1 km 2.508193864756265 2.508193864756265
OK   dBT 5 km 11.455202740120455 11.455202740120399
OK   degenerate tau (1 km) 0.5376165784494953 0.5376165784494724
OK   O2 above 1 km 0.8668778997501816 0.8668778997501816
OK   O2 above 3 km 0.6514390575310556 0.6514390575310556
OK   O2 above 5 km 0.4895416595569531 0.4895416595569531
OK   O2 above 8 km 0.31890655732397044 0.31890655732397044
The 5 km layer gives a larger dBT by a factor of 4.57


## Study guide v2 — Part A, worked examples, mock exam Q9–Q10

In [9]:
N=20e-9*2.5e19; tau=1.5e-19*1e5*N; chk('NO2 tau', tau, V['a_no2']['tau'], 1e-9); chk('NO2 t', math.exp(-tau), V['a_no2']['t'], 1e-9)
chk('Q9c tau', 2e-20*5e13*1e5, V['q9c']['tau'], 1e-9); chk('Q10c D', 2e-19*1e16, V['q10c']['D'], 1e-9)
chk('plume 72 km / 18 km/h', 72/18, V['plume72']['tau_h'])
du=300*2.687e16; am=1/math.cos(math.radians(60))+1
for lam,sg in [('310',1e-19),('330',1e-20)]: chk('ozone T '+lam, math.exp(-sg*du*am), V['ozone'][lam]['T'], 1e-9)
chk('megacity VCD', 2.4e16/1.2, V['megacity']['vcd'], 1e-9); chk('limb path', math.sqrt(8*6371*1), V['limb']['L'], 1e-9)
K=np.array([[1.,1.]]); A2=np.eye(2)@K.T@np.linalg.inv(K@np.eye(2)@K.T+0.01)@K; chk('two-layer DOFS', np.trace(A2), V['two_layer']['dofs'], 1e-9)
chk('alpha smoke', -math.log(0.8/0.2)/math.log(440/870), V['ang_smoke']['alpha'], 1e-9); chk('alpha dust', -math.log(0.5/0.45)/math.log(440/870), V['ang_dust']['alpha'], 1e-9)
chk('x droplet visible', 2*math.pi*10/0.55, V['droplet']['x_vis'], 1e-9); chk('x droplet radar', 2*math.pi*10/3000, V['droplet']['x_radar'], 1e-9)
chk('O2 above 6 km', math.exp(-6/8), V['o2_scale8']['6'], 1e-9)
amu=1.66053907e-27; dop=2927/C*math.sqrt(2*math.log(2)*KB*250/(16.04*amu)); chk('Doppler CH4', dop, V['lines']['doppler_ch4_2927'], 1e-9)
b4,_=bt(310,4,0.7); b1_,_=bt(310,1,0.7); chk('TIR 310 K 4 km', 310-b4, V['tir310']['z4']['dBT'], 1e-9)
print('NOTE: the guide quotes ~7 K (4 km) and ~2 K (1 km); our blackbody-layer model gives', round(310-b4,1), 'and', round(310-b1_,1), 'K (orders of magnitude agree, trend identical)')

OK   NO2 tau 0.0075 0.0075
OK   NO2 t 0.9925280548191384 0.9925280548191384
OK   Q9c tau 0.09999999999999999 0.09999999999999999
OK   Q10c D 0.002 0.002
OK   plume 72 km / 18 km/h 4.0 4.0
OK   ozone T 310 0.08907291277364751 0.08907291277364751
OK   ozone T 330 0.785189648446718 0.785189648446718
OK   megacity VCD 2e+16 2e+16
OK   limb path 225.76093550479453 225.76093550479453
OK   two-layer DOFS 0.9950248756218907 0.9950248756218907
OK   alpha smoke 2.0335290771176413 2.0335290771176413
OK   alpha dust 0.1545513551661696 0.1545513551661696
OK   x droplet visible 114.23973285781065 114.23973285781065
OK   x droplet radar 0.020943951023931952 0.020943951023931952
OK   O2 above 6 km 0.4723665527410147 0.4723665527410147
OK   Doppler CH4 0.004138230358209229 0.004138230358209229
OK   TIR 310 K 4 km 12.261068084763224 12.261068084763167
NOTE: the guide quotes ~7 K (4 km) and ~2 K (1 km); our blackbody-layer model gives 12.3 and 3.2 K (orders of magnitude agree, trend identical)


**Conclusion:** every comparison prints `OK` (a `FAIL` would stop the cell with an `assert`). The JavaScript file `_shared/calc.js` is tested separately by `tests/test_calc.js` (466 comparisons against the same `verified.json`).